# 01 — Data exploration

load COIL-20, shape checks, sample grid, split sizes, S_w rank

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.config import IMG_SIZE, OFFSETS, RESULTS_DIR, STEP
from src.data import load_coil20, make_split

## Load

In [ ]:
X, y, pose = load_coil20()
X_raw, _, _ = load_coil20(normalise=False)
print(X.shape, y.shape, pose.shape)

## Sanity checks

In [ ]:
assert X.shape == (1440, IMG_SIZE * IMG_SIZE)
assert len(np.unique(y)) == 20 and (np.bincount(y) == 72).all()
assert np.allclose(np.linalg.norm(X, axis=1), 1) and np.allclose(X.mean(axis=1), 0)

train_sets = []
for off in OFFSETS:
    tr, te = make_split(pose, off)
    assert len(tr) == 240 and len(te) == 1200
    assert len(np.intersect1d(tr, te)) == 0
    assert (np.bincount(y[tr], minlength=20) == 12).all()
    train_sets.append(tr)
assert len(np.unique(np.concatenate(train_sets))) == 1440   # disjoint training sets cover everything
print("all checks passed")

## Sample grid (5 objects x 6 poses)

In [ ]:
objs, poses = [0, 4, 8, 12, 16], [0, 12, 24, 36, 48, 60]
fig, axes = plt.subplots(len(objs), len(poses), figsize=(9, 7.5))
for r, k in enumerate(objs):
    for c, p in enumerate(poses):
        i = np.flatnonzero((y == k) & (pose == p))[0]
        ax = axes[r, c]
        ax.imshow(X_raw[i].reshape(IMG_SIZE, IMG_SIZE), cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0: ax.set_title(f"{5 * p}°")
        if c == 0: ax.set_ylabel(f"obj {k + 1}")
fig.tight_layout()
(RESULTS_DIR / "figures").mkdir(parents=True, exist_ok=True)
fig.savefig(RESULTS_DIR / "figures" / "samples.png", dpi=300)